# Random forests: Breiman (2001)

In 2001 the most accurate off-the-shelf classifier was AdaBoost, which grows each tree on the cases its predecessors got wrong. Leo Breiman's "Random Forests" proposed something simpler: grow many deep trees, each on a bootstrap sample of the data and each choosing among a few random inputs at every split, and let them vote. Nothing adapts, nothing is pruned, and the forest estimates its own error from the cases each tree never saw. On 20 data sets the forests matched or beat AdaBoost (Table 2, p. 11), and the idea became one of the most used methods in statistics and machine learning. The forests, Breiman's AdaBoost, Table 2 and his measure of variable importance are replicated here.

The data are 15 of Breiman's 16 real data sets, in PollisDatasets as he described them (Table 1, p. 10), four simulated sets generated as he did, and the zip-code digits, downloaded only if you agree (Section 5). The trees are grown with DecisionTree.jl; the forests, their out-of-bag errors, Breiman's AdaBoost with its pruning, and the variable importance are written out in a few lines each.

In [ ]:
Pollis.packages("DecisionTree", "StatsBase", "PollisDatasets", "Plots", "CodecZlib", "ProgressLogging")

## Set-up

Table 2 averages 100 random runs per data set, so its values are compared by `compare`: each Julia average with its standard error, rounded beside the published value, and whether the two differ by less than two standard errors of a difference of two such averages, Breiman's being one too. `repetitions` sets the number of runs: 100 as in the paper, which takes about an hour on one thread and a few minutes on ten. Every run has its own seed, so the results do not depend on the number of threads, and the status bar shows how far each data set has got; to use more, set **julia.NumThreads** to `auto` in the settings and restart the kernel. With 20 repetitions, the picture is the same in a fifth of the time, with standard errors √5 times larger.

`inputs` reads a data set as Breiman used it: the inputs as a matrix, a yes vote as 1, and the class. Breiman does not say how he treated the few missing values of breast cancer, votes and soybean; here each is replaced by the median of its input, the rough fix of R's randomForest.

In [ ]:
using DecisionTree, StatsBase, PollisDatasets, Plots, CodecZlib, ProgressLogging, DelimitedFiles, Downloads, Printf, Random, Statistics
using DecisionTree: Leaf, Node

"""
	compare(title, rows)

Print each Julia error rate, in percent, with its standard error, rounded beside the published value, and whether they differ by less than
two standard errors of a difference (√2 se: the published value is an estimate of the same precision). Each row is `(label, value, se, published)`.
"""
function compare(title, rows)
	@printf("\n%s\n%-16s%10s%8s%10s%11s\n", title, "", "Julia", "se", "rounded", "published")
	for (label, value, se, published) in rows
		z = (value - parse(Float64, published)) / (sqrt(2) * se)
		@printf("%-16s%10.3f%8.3f%10.1f%11s  %s\n", label, value, se, value, published, abs(z) < 2 ? "within noise" : @sprintf("DIFFERENT (%.1f se)", z))
	end
end

repetitions = 100  # runs per data set, as in the paper
println("threads: ", Threads.nthreads())

"""Run `f(rng)` for runs 1:n, each with its own seed, on all threads, with a progress bar named `name`."""
function runs(f, n; name="runs")
	r = Vector{Any}(undef, n)
	finished = Threads.Atomic{Int}(0)
	@withprogress name=name begin
		Threads.@threads for i in 1:n
			r[i] = f(Xoshiro(i))
			@logprogress (Threads.atomic_add!(finished, 1) + 1) / n
		end
	end
	r
end

const CLASS = Dict("glass" => :type, "breastwisconsin" => :class, "pima" => :diabetes, "sonar" => :class, "vowel" => :vowel,
	"ionosphere" => :class, "vehicle" => :class, "soybean" => :disease, "germancredit" => :credit, "image" => :class,
	"ecoli" => :site, "votes" => :party, "liver" => :selector, "letters" => :letter, "satimage" => :class)
const NOTINPUT = [:id, :protein, :speaker, :segment, :set]  # identifiers and the train/test marker

"""The inputs of data set `id` as a matrix, missing values replaced by the median of their input, and the classes numbered 1, 2, …"""
function inputs(id)
	d = dataset(id)
	cols = [k for k in keys(d) if k ∉ NOTINPUT && k != CLASS[id]]
	X = reduce(hcat, [begin
		c = [v isa AbstractString ? Float64(v == "y") : v for v in d[k]]  # votes: yes = 1, no = 0
		m = median(skipmissing(c))
		Float64[ismissing(v) ? m : v for v in c]
	end for k in cols])
	labels = string.(d[CLASS[id]])
	names = sort(unique(labels))
	X, indexin(labels, names), length(names)
end

## 1. The data sets

Table 1 (p. 10) lists 13 small data sets, 3 larger ones with their own training and test sets, and 4 simulated ones. All but zip-code and the simulated ones are in PollisDatasets, as Breiman described them: the German credit data in the 24 numeric inputs of its StatLog version, the vowels with Deterding's 10 inputs, and liver with its selector column as the class. Soybean has 683 cases, not 685: Breiman's own earlier paper on AdaBoost lists 683 (Breiman 1998, Table 4).

In [ ]:
small = ["glass", "breastwisconsin", "pima", "sonar", "vowel", "ionosphere", "vehicle", "soybean", "germancredit", "image", "ecoli", "votes", "liver"]
table1 = Dict("glass" => (214, 9, 6), "breastwisconsin" => (699, 9, 2), "pima" => (768, 8, 2), "sonar" => (208, 60, 2), "vowel" => (990, 10, 11),
	"ionosphere" => (351, 34, 2), "vehicle" => (846, 18, 4), "soybean" => (685, 35, 19), "germancredit" => (1000, 24, 2), "image" => (2310, 19, 7),
	"ecoli" => (336, 7, 8), "votes" => (435, 16, 2), "liver" => (345, 6, 2), "letters" => (20000, 16, 26), "satimage" => (6435, 36, 6))
data = Dict(id => inputs(id) for id in keys(table1))
@printf("%-16s%16s%16s%16s\n", "", "cases", "inputs", "classes")
for id in [small; "letters"; "satimage"]
	X, y, C = data[id]; n, M, K = table1[id]
	@printf("%-16s%16s%16s%16s\n", id, "$(size(X, 1)) ($n)", "$(size(X, 2)) ($M)", "$C ($K)")
end
println("(in brackets: Table 1; letters and sat-images, training plus test cases)")

## 2. Forest-RI

A random forest with random input selection, Forest-RI (p. 9), grows each tree on a bootstrap sample of the training set: n cases drawn with replacement. At each node it draws F inputs at random and splits on the best of them, by the Gini criterion of CART, and it grows each tree to the end, unpruned. The forest classifies by majority vote.

About a third of the cases are left out of each bootstrap sample. Run down the trees that did not see them, they give the **out-of-bag** error: the error of the forest's vote on each case, counting only those trees, an estimate of the test error that needs no test set (Section 3.1, p. 11). The same cases give the error of each tree on its own, Table 2's "One tree" column. `forest` returns the trees and both errors.

DecisionTree.jl's `build_tree` grows the trees; `classify` runs cases down a tree without recursion, since the deepest trees overflow the stack of DecisionTree.jl's recursive `apply_tree`.

In [ ]:
"""Run each row of `X` down tree `t`."""
function classify(t, X::AbstractMatrix)
	t isa DecisionTree.Root && (t = t.node)
	map(eachrow(X)) do x
		s = t
		while s isa Node
			s = x[s.featid] < s.featval ? s.left : s.right
		end
		s.majority
	end
end

"""The class with the most votes in each row of `votes`."""
winner(votes) = [argmax(r) for r in eachrow(votes)]

"""
	forest(X, y, C, F, ntree, rng)

Forest-RI: `ntree` unpruned Gini trees, each on a bootstrap sample, splitting each node on the best of `F` random inputs.
Returns the trees, the out-of-bag error of the forest, and the average out-of-bag error of its trees.
"""
function forest(X, y, C, F, ntree, rng)
	n = length(y)
	votes = zeros(n, C)
	trees = Vector{Any}(undef, ntree)
	single = 0.0
	for b in 1:ntree
		boot = rand(rng, 1:n, n)
		trees[b] = build_tree(y[boot], X[boot, :], F, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)
		oob = setdiff(1:n, boot)
		p = classify(trees[b], X[oob, :])
		single += mean(p .!= y[oob])
		for (i, c) in zip(oob, p)
			votes[i, c] += 1
		end
	end
	seen = vec(sum(votes, dims=2)) .> 0
	trees, mean(winner(votes[seen, :]) .!= y[seen]), single / ntree
end

"""The (weighted) vote of `trees` on each row of `X`."""
function vote(trees, X, C; weights=ones(length(trees)))
	votes = zeros(size(X, 1), C)
	for (t, w) in zip(trees, weights)
		for (i, c) in enumerate(classify(t, X))
			votes[i, c] += w
		end
	end
	winner(votes)
end

X, y, C = data["glass"]
for F in (1, 4)
	trees, oob, one = forest(X, y, C, F, 100, Xoshiro(1))
	@printf("glass, F = %d: out-of-bag error of the forest %.1f%%, of a single tree %.1f%%\n", F, 100oob, 100one)
end

## 3. AdaBoost, as Breiman ran it

Breiman's AdaBoost column is not the deterministic AdaBoost of Freund and Schapire (1996) that the paper describes (p. 3). Its values are those of **arc-fs**, the version of Breiman's earlier paper on arcing (Breiman 1998, Table 5: breast cancer 3.2, ionosphere 6.4, diabetes 26.6, glass 22.0, letters 3.4, satellite 8.8, digits 6.2; Table 3: waveform 17.8, twonorm 4.9). It is written out here from that paper (pp. 6-7).

It keeps a probability p(n) on each training case, equal at the start. At step k it draws a training set T′ from the cases with these probabilities, grows a CART tree on it, and prunes the tree on a second draw T″. With d(n) = 1 for the cases of the full training set the tree misclassifies,

εₖ = Σ p(n) d(n),  βₖ = (1 − εₖ) / εₖ,  p(n) ← p(n) βₖ^d(n), normalised,

so the misclassified cases weigh more in the next draw. The trees vote with weights log βₖ. When εₖ reaches ½, or is 0, the probabilities are set equal again and the construction restarts.

CART prunes by **cost complexity** (Breiman et al. 1984, Ch. 3): it repeatedly collapses the weakest link, the internal node t with the smallest g(t) = (R(t) − R(Tₜ)) / (|Tₜ| − 1), where R(t) is the number of training cases misclassified at t as a leaf, R(Tₜ) by the branch below it, and |Tₜ| the leaves of the branch. Of this sequence of subtrees, `prune` keeps the one with the fewest errors on T″, the smallest on ties.

In [ ]:
caselabels(t::Leaf) = t.values  # the training classes that reached each leaf
caselabels(t::Node) = vcat(caselabels(t.left), caselabels(t.right))
misclassified(v) = length(v) - maximum(values(countmap(v)))  # R(t): errors as a leaf
nleaves(t::Leaf) = 1
nleaves(t::Node) = nleaves(t.left) + nleaves(t.right)
branchmisclassified(t::Leaf) = misclassified(t.values)
branchmisclassified(t::Node) = branchmisclassified(t.left) + branchmisclassified(t.right)
g(t::Node) = (misclassified(caselabels(t)) - branchmisclassified(t)) / (nleaves(t) - 1)

"""The smallest g(t) over the internal nodes of `t`: the weakest link."""
weakest(t::Leaf) = Inf
weakest(t::Node) = min(g(t), weakest(t.left), weakest(t.right))

"""Collapse to a leaf every internal node with g(t) at most α."""
cut(t::Leaf, α) = t
cut(t::Node, α) = g(t) <= α + 1e-12 ? (v = caselabels(t); Leaf(mode(v), v)) : Node(t.featid, t.featval, cut(t.left, α), cut(t.right, α))

"""The subtree of the cost-complexity sequence of `t` with the fewest errors on (X, y), the smallest on ties."""
function prune(t, X, y)
	best, fewest = t, sum(classify(t, X) .!= y)
	while t isa Node
		t = cut(t, weakest(t))
		e = sum(classify(t, X) .!= y)
		if e <= fewest
			best, fewest = t, e
		end
	end
	best
end

"""
	arcfs(X, y, K, rng)

Breiman's (1998) arc-fs: `K` steps of adaptive resampling with Freund and Schapire's update, each tree pruned on a second draw.
Returns the trees and their weights log β. A restart drops the tree of its step.
"""
function arcfs(X, y, K, rng)
	n = length(y)
	p = fill(1 / n, n)
	trees, weights = Any[], Float64[]
	for k in 1:K
		draw1, draw2 = sample(rng, 1:n, Weights(p), n), sample(rng, 1:n, Weights(p), n)
		full = build_tree(y[draw1], X[draw1, :], 0, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)
		t = prune(full.node, X[draw2, :], y[draw2])
		d = classify(t, X) .!= y
		ε = sum(p[d])
		if ε >= 1 / 2 || ε == 0
			fill!(p, 1 / n)  # restart
			continue
		end
		β = (1 - ε) / ε
		p .*= β .^ d
		p ./= sum(p)
		push!(trees, t)
		push!(weights, log(β))
	end
	trees, weights
end

## 4. Table 2: the small data sets

On each small data set Breiman set aside a random 10% as a test set. On the rest he grew two forests of 100 trees, one with F = 1 and one with F = int(log₂ M + 1), M the number of inputs, and took the test error of the one with the lower out-of-bag error: the **Selection** column. **Single input** is the test error of the forest with F = 1, and **One tree** the out-of-bag error of the single trees of the selected forest. AdaBoost combined 50 trees on the same split. This was repeated 100 times and the errors averaged (pp. 10-11).

In [ ]:
"""Breiman's four errors, in percent, for a forest of `ntree` trees and `K` steps of arc-fs trained on `train`, tested on `test`."""
function table2(X, y, C, train, test, rng; ntree=100, K=50)
	F = floor(Int, log2(size(X, 2)) + 1)
	t1, oob1, one1 = forest(X[train, :], y[train], C, 1, ntree, rng)
	tF, oobF, oneF = forest(X[train, :], y[train], C, F, ntree, rng)
	single = mean(vote(t1, X[test, :], C) .!= y[test])
	selection = oobF < oob1 ? mean(vote(tF, X[test, :], C) .!= y[test]) : single
	trees, weights = arcfs(X[train, :], y[train], K, rng)
	adaboost = mean(vote(trees, X[test, :], C; weights=weights) .!= y[test])
	100 .* [adaboost, selection, single, oobF < oob1 ? oneF : one1]
end

"""One run on a small data set: a random 10% set aside as the test set."""
function smallrun(X, y, C, rng)
	n = length(y)
	test = randperm(rng, n)[1:round(Int, n / 10)]
	table2(X, y, C, setdiff(1:n, test), test, rng)
end

published = Dict("glass" => ("22.0", "20.6", "21.2", "36.9"), "breastwisconsin" => ("3.2", "2.9", "2.7", "6.3"), "pima" => ("26.6", "24.2", "24.3", "33.1"),
	"sonar" => ("15.6", "15.9", "18.0", "31.7"), "vowel" => ("4.1", "3.4", "3.3", "30.4"), "ionosphere" => ("6.4", "7.1", "7.5", "12.7"),
	"vehicle" => ("23.2", "25.8", "26.4", "33.1"), "germancredit" => ("23.5", "24.4", "26.2", "33.3"), "image" => ("1.6", "2.1", "2.7", "6.4"),
	"ecoli" => ("14.8", "12.8", "13.0", "24.5"), "votes" => ("4.8", "4.1", "4.6", "7.4"), "liver" => ("30.7", "25.1", "24.7", "40.6"),
	"letters" => ("3.4", "3.5", "4.7", "19.8"), "satimage" => ("8.8", "8.6", "10.5", "17.2"), "zip" => ("6.2", "6.3", "7.8", "20.6"),
	"waveform" => ("17.8", "17.2", "17.3", "34.0"), "twonorm" => ("4.9", "3.9", "3.9", "24.7"), "threenorm" => ("18.8", "17.5", "17.5", "38.4"),
	"ringnorm" => ("6.9", "4.9", "4.9", "25.7"))
columns = ("AdaBoost", "Selection", "Single input", "One tree")
results = Dict{String, Any}()  # data set => (averages, standard errors)

function report(id, r)
	R = reduce(hcat, r)'
	m, se = vec(mean(R, dims=1)), size(R, 1) > 1 ? vec(std(R, dims=1)) ./ sqrt(size(R, 1)) : fill(NaN, 4)
	results[id] = (m, se)
	compare("$id (Table 2, p. 11), $(size(R, 1)) runs", [(columns[j], m[j], se[j], published[id][j]) for j in 1:4])
end

for id in filter(!=("soybean"), small)  # soybean is not in Table 2 (Section 8)
	X, y, C = data[id]
	report(id, runs(rng -> smallrun(X, y, C, rng), repetitions; name=id))
end

## 5. Table 2: the larger data sets

Letters, sat-images and zip-code have their own training and test sets, so each is one run: forests of 100 trees (200 for zip-code) and AdaBoost of 50 trees (100 for zip-code) (p. 11). With a single run there is no average to take a standard error of; the standard error shown is the binomial one of a test error on that many cases, sqrt(e(1 − e)/n). arc-fs grows and prunes every tree on all the inputs, so AdaBoost takes several minutes on letters.

In [ ]:
"""One run on a data set with its own test set; the standard errors are binomial."""
function largerun(id, X, y, C, train, test; ntree=100, K=50)
	e = @withprogress name=id table2(X, y, C, train, test, Xoshiro(2001); ntree=ntree, K=K)
	results[id] = (e, sqrt.(e .* (100 .- e) ./ length(test)))
	compare("$id (Table 2, p. 11), $(length(train)) training and $(length(test)) test cases", [(columns[j], e[j], results[id][2][j], published[id][j]) for j in 1:4])
end

X, y, C = data["letters"]
largerun("letters", X, y, C, 1:15000, 15001:20000)  # the first 15,000 cases train: the papers do not say which
X, y, C = data["satimage"]
set = dataset("satimage").set
largerun("satimage", X, y, C, findall(==("train"), set), findall(==("test"), set))

### The zip-code digits

The zip-code data are 9,298 handwritten digits scanned from envelopes by the U.S. Postal Service, each as 16 × 16 grey levels: 256 inputs. They are not in PollisDatasets, so the next cell downloads them, 2.3 MB, from the site of Hastie, Tibshirani and Friedman's *The Elements of Statistical Learning*, and keeps them in ~/.pollis/data/zipcode. It does so only if you set `download_zipcode = true` and run it; otherwise the zip-code row stays empty. The forests of 200 trees on 256 inputs, and above all the 100 pruned AdaBoost trees, take a while.

In [ ]:
download_zipcode = false  # set to true and run this cell to download the zip-code data (2.3 MB, once)
zipdir = joinpath(homedir(), ".pollis", "data", "zipcode")
if download_zipcode
	mkpath(zipdir)
	for f in ("zip.train.gz", "zip.test.gz")
		isfile(joinpath(zipdir, f)) || Downloads.download("https://hastie.su.domains/ElemStatLearn/datasets/$f", joinpath(zipdir, f))
	end
end

In [ ]:
"""A zip-code file: the digit, then the 256 grey levels, one case per line."""
function zipread(f)
	d = readdlm(GzipDecompressorStream(open(f)), Float64)[:, 1:257]
	d[:, 2:end], Int.(d[:, 1]) .+ 1
end

if isfile(joinpath(zipdir, "zip.test.gz"))
	X1, y1 = zipread(joinpath(zipdir, "zip.train.gz"))
	X2, y2 = zipread(joinpath(zipdir, "zip.test.gz"))
	largerun("zip", vcat(X1, X2), vcat(y1, y2), 10, 1:length(y1), length(y1)+1:length(y1)+length(y2); ntree=200, K=100)
else
	println("zip-code: not downloaded")
end

## 6. Table 2: the simulated data sets

The four simulated data sets come from Breiman's (1998) arcing paper (p. 7), with two classes, or three for waveform, equally likely:

- **twonorm**: 20 inputs, normal with identity covariance and mean (a, a, …, a) for class 1, (−a, −a, …, −a) for class 2, a = 2/√20.
- **threenorm**: class 1 is an equal mixture of the normals with means (a, …, a) and (−a, …, −a), class 2 the normal with mean (a, −a, a, −a, …), a = 2/√20.
- **ringnorm**: class 1 is normal with mean 0 and covariance 4 times the identity, class 2 normal with mean (a, …, a) and the identity, a = 1/√20.
- **waveform** (Breiman et al. 1984, p. 49): 21 inputs, each class a random mixture u h + (1 − u) h′ of two of three triangular waves, plus independent N(0, 1) noise: class 1 mixes h₁ and h₂, class 2 h₁ and h₃, class 3 h₂ and h₃, with u uniform on (0, 1), h₁(i) = max(6 − |i − 11|, 0), h₂(i) = h₁(i − 4) and h₃(i) = h₁(i + 4).

Each of 50 runs draws a new training set of 300 cases and a test set of 3,000, and grows forests of 100 trees and AdaBoost of 50 (p. 11).

In [ ]:
function twonorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	randn(rng, n, 20) .+ a .* ifelse.(y .== 1, 1, -1), y
end

function threenorm(n, rng)
	a = 2 / sqrt(20)
	y = rand(rng, 1:2, n)
	alternating = [isodd(j) ? a : -a for j in 1:20]'
	X = randn(rng, n, 20)
	for i in 1:n
		X[i, :] .+= y[i] == 2 ? alternating' : fill(rand(rng, Bool) ? a : -a, 20)
	end
	X, y
end

function ringnorm(n, rng)
	a = 1 / sqrt(20)
	y = rand(rng, 1:2, n)
	X = randn(rng, n, 20)
	ifelse.(y .== 1, 2 .* X, X .+ a), y
end

function waveform(n, rng)
	h1 = [max(6 - abs(i - 11), 0) for i in 1:21]
	h2 = [max(6 - abs(i - 4 - 11), 0) for i in 1:21]
	h3 = [max(6 - abs(i + 4 - 11), 0) for i in 1:21]
	mixtures = ((h1, h2), (h1, h3), (h2, h3))
	y = rand(rng, 1:3, n)
	X = randn(rng, n, 21)
	for i in 1:n
		u = rand(rng)
		h, h′ = mixtures[y[i]]
		X[i, :] .+= u .* h .+ (1 - u) .* h′
	end
	X, y
end

for (id, simulate, C) in (("waveform", waveform, 3), ("twonorm", twonorm, 2), ("threenorm", threenorm, 2), ("ringnorm", ringnorm, 2))
	report(id, runs(50; name=id) do rng
		X, y = simulate(300, rng)
		Xt, yt = simulate(3000, rng)
		table2([X; Xt], [y; yt], C, 1:300, 301:3300, rng)
	end)
end

## 7. The whole of Table 2

Each of Breiman's values against its replication. The forests beat AdaBoost on most sets in both, and the single trees, each grown on one random input at a time, are far worse than the forests they form: the point of the paper.

In [ ]:
plot(xlabel="Breiman (2001), Table 2, %", ylabel="Julia, %", legend=:topleft, size=(560, 520), aspect_ratio=:equal)
for (j, name) in enumerate(columns)
	ids = sort(collect(keys(results)))
	scatter!([parse(Float64, published[id][j]) for id in ids], [results[id][1][j] for id in ids], label=name, markersize=4)
end
plot!([0, 42], [0, 42], color=:gray, linestyle=:dash, label="")

## 8. Soybean

Soybean is in Table 1 but not in Table 2. Breiman reports it in Section 6 (p. 15), on data sets with many categorical inputs: with Forest-RI and F = 12, a test error of 5.3%, against 5.8% for AdaBoost. The procedure is Table 2's, with the one forest.

In [ ]:
X, y, C = data["soybean"]
r = reduce(hcat, runs(repetitions; name="soybean") do rng
	n = length(y)
	test = randperm(rng, n)[1:round(Int, n / 10)]
	train = setdiff(1:n, test)
	trees, _, _ = forest(X[train, :], y[train], C, 12, 100, rng)
	arc, weights = arcfs(X[train, :], y[train], 50, rng)
	100 .* [mean(vote(arc, X[test, :], C; weights=weights) .!= y[test]), mean(vote(trees, X[test, :], C) .!= y[test])]
end)'
compare("soybean (Section 6, p. 15), $(size(r, 1)) runs", [(label, mean(r[:, j]), std(r[:, j]) / sqrt(size(r, 1)), p) for (j, label, p) in ((1, "AdaBoost", "5.8"), (2, "Forest-RI, F=12", "5.3"))])

## 9. Which inputs matter

To see which inputs a forest relies on, Breiman (Section 10, pp. 19-21) permutes the values of input m among the out-of-bag cases of each tree, after growing it, and runs them down the tree again. The out-of-bag vote with input m so noised up is compared with the true classes, and the percent increase over the out-of-bag error with all inputs intact measures the importance of m. He grows one forest of 1,000 trees, with no test set.

On diabetes, with F = 1, input 2 (plasma glucose) stands out, followed by 8 (age) and 6 (body mass) (Figure 4, p. 20). His figures have no printed values, so they are compared by eye; the test errors of forests on chosen inputs are printed (p. 20), from the 100 runs of Table 2: 29.7% with input 2 alone, 23.1% with all eight, 29.4% with inputs 2 and 8, and 26.4% with inputs 2 and 6. Input 8 adds nothing once 2 is in: it carries much the same information.

In [ ]:
"""
	importance(X, y, C, F, ntree, rng)

Section 10: the percent increase in the out-of-bag error when each input in turn is permuted among the out-of-bag cases of every tree.
"""
function importance(X, y, C, F, ntree, rng)
	n, M = size(X)
	votes, noised = zeros(n, C), zeros(n, C, M)
	for b in 1:ntree
		boot = rand(rng, 1:n, n)
		t = build_tree(y[boot], X[boot, :], F, -1, 1, 2, 0.0; loss=DecisionTree.util.gini, rng=rng, impurity_importance=false)
		oob = setdiff(1:n, boot)
		for (i, c) in zip(oob, classify(t, X[oob, :]))
			votes[i, c] += 1
		end
		for m in 1:M
			Xm = X[oob, :]
			Xm[:, m] = shuffle(rng, Xm[:, m])
			for (i, c) in zip(oob, classify(t, Xm))
				noised[i, c, m] += 1
			end
		end
	end
	seen = vec(sum(votes, dims=2)) .> 0
	base = mean(winner(votes[seen, :]) .!= y[seen])
	[100 * (mean(winner(noised[seen, :, m]) .!= y[seen]) - base) / base for m in 1:M]
end

"""The Selection error of Table 2 for a forest on the inputs `vars` only, over `repetitions` runs of a 10% test set."""
function selectionerror(X, y, C, vars)
	r = runs(repetitions; name="inputs $(join(vars, ", "))") do rng
		n = length(y)
		test = randperm(rng, n)[1:round(Int, n / 10)]
		train = setdiff(1:n, test)
		Xv = X[:, vars]
		F = floor(Int, log2(length(vars)) + 1)
		t1, oob1, _ = forest(Xv[train, :], y[train], C, 1, 100, rng)
		tF, oobF, _ = forest(Xv[train, :], y[train], C, F, 100, rng)
		100mean(vote(oobF < oob1 ? tF : t1, Xv[test, :], C) .!= y[test])
	end
	mean(r), std(r) / sqrt(length(r))
end

X, y, C = data["pima"]
diabetes = importance(X, y, C, 1, 1000, Xoshiro(10))
compare("diabetes, forests on chosen inputs (p. 20)", [(label, selectionerror(X, y, C, vars)..., p) for (label, vars, p) in
	(("input 2", [2], "29.7"), ("all inputs", 1:8, "23.1"), ("inputs 2 and 8", [2, 8], "29.4"), ("inputs 2 and 6", [2, 6], "26.4"))])
bar(1:8, diabetes, xticks=1:8, xlabel="input", ylabel="percent increase in error", label="", title="diabetes (Figure 4, p. 20)", size=(560, 330))

On the votes of the 435 members of the U.S. House of Representatives on 16 issues in 1984, Breiman used F = 5, the setting with the lowest error, and 1,000 trees (p. 21). Input 4, the vote on freezing physicians' fees, stands out: noising it up triples the error (Figure 6). A forest on input 4 alone has a test error of 4.3%, about the same as with all 16.

In [ ]:
X, y, C = data["votes"]
votes = importance(X, y, C, 5, 1000, Xoshiro(10))
compare("votes, a forest on input 4 alone (p. 21)", [("input 4", selectionerror(X, y, C, [4])..., "4.3")])
bar(1:16, votes, xticks=1:16, xlabel="input", ylabel="percent increase in error", label="", title="votes (Figure 6, p. 21)", size=(560, 330))

## 10. What differs, and why

With 100 runs on 10 threads, 63 of the 82 values agree with Breiman's within noise. Among 82 comparisons a few should land beyond two standard errors by chance alone; the rest are listed here.

- **One tree (11 of 19 beyond noise).** Each value averages the out-of-bag errors of thousands of single trees, so its standard error is tiny (0.03 to 0.5) and any difference in how one unpruned tree is grown shows: how ties between splits are broken, where a split point is placed between two values. The differences go both ways, by 0.2 to 2 points, while the forests built from these trees agree. Vowel (36.1 against 30.4) also depends on the reading of "the best setting": with F = 1 every run gives about 37, with F = 4 about 26; Breiman's 30.4 lies between, as if his runs selected F = 4 more often than these do.
- **AdaBoost on breast cancer (3.8 against 3.2) and diabetes (25.1 against 26.6).** arc-fs depends on details the arcing paper leaves open: whether the tree of a step that restarts joins the vote (here it does not), and how its CART chose among equally good pruned subtrees (here the smallest). Not explained.
- **The simulated sets.** Twonorm and ringnorm come out 0.2 to 0.6 points below Breiman, threenorm's single trees 2 points below. The generators follow the arcing paper's description (p. 7), which leaves little open; the difference may be in the trees. Not explained.
- **Liver's Selection (27.3 against 25.1) and diabetes on inputs 2 and 6 (28.1 against 26.4).** At about 2 standard errors: within what chance gives among 82 comparisons.
- **The data.** Soybean has 683 cases, not the 685 of Table 1. Its categorical inputs are split as ordered codes: DecisionTree.jl has no splits on subsets of categories, which Breiman's program had. The few missing values are replaced by medians; Breiman does not say what he did.
- **Variable importance.** Figures 4 and 6 have no printed values; the ordering agrees: inputs 2, 8 and 6 on diabetes, input 4 far ahead on votes. The size of the rises depends on the forest's own error, which differs from run to run.

**Now in Compose › New Methods.** DecisionTree.jl grows the trees, but its forests have no out-of-bag error and no permutation importance, and no Julia package has arc-fs or CART's cost-complexity pruning on a test set. Two New Methods notebooks write them out as documented functions, tested against Breiman's published errors: **Out-of-bag error and variable importance for random forests** (Table 2 and Section 10 of the 2001 paper) and **arc-fs: adaptive resampling with pruned trees** (Table 2). Open them from **Compose › New Methods**. Forest-RC, the forests on random linear combinations of inputs (Table 3, p. 13; Figure 5, p. 20), is not yet among them.

## References

- Breiman, L. (1996). Bias, variance, and arcing classifiers. Technical Report 460, Statistics Department, University of California, Berkeley.
- Breiman, L. (1998). Arcing classifiers. *The Annals of Statistics* 26(3), 801-849. doi:10.1214/aos/1024691079
- Breiman, L. (2001). Random forests. *Machine Learning* 45(1), 5-32. doi:10.1023/A:1010933404324
- Breiman, L., Friedman, J. H., Olshen, R. A. and Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.
- Freund, Y. and Schapire, R. E. (1996). Experiments with a new boosting algorithm. *Machine Learning: Proceedings of the Thirteenth International Conference*, 148-156.
- Hastie, T., Tibshirani, R. and Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer: the zip-code data.
- The UCI Machine Learning Repository, for the other real data sets: each is cited, with its DOI, by `datasetinfo` in PollisDatasets.